# Compare & validate HOL4 → Lean translations (Gemini 2.5)

This notebook pairs HOL4 theory scripts (`xxxScript.sml`) with the corresponding generated Lean files (`xxx.lean`) and asks **Gemini 2.5** to spot likely translation mistakes.

**Mapping rule**: `xxxScript.sml` (in HOL4) ↔ `xxx.lean` (in Lean).

Paths used:
- Lean: `E:/NUS/mcomp/Dissertation/cakeML/cakeml/lean_type_sound/LeanTypeSound`
- HOL4: `E:/NUS/mcomp/Dissertation/cakeML/cakeml/lean_type_sound/hol4-ultramin`

## Prerequisites

1. Set your Gemini API key in the environment (PowerShell):
   - `$env:GEMINI_API_KEY = "..."`

2. Ensure you have read access to the two folders above.

Notes:
- The notebook writes results to `compare_validate_results.jsonl` in this workspace so you can resume without re-calling the API.
- By default it validates only a small number of pairs; adjust `MAX_FILES` if you want more.

In [1]:
from __future__ import annotations

import os
import json
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

# --- Configuration (edit if needed) ---
LEAN_ROOT = Path(r"D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\LeanTypeSound")
HOL4_ROOT = Path(r"D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\hol4-ultramin")

# Where to write/append results in this repo
RESULTS_JSONL = Path('compare_validate_results.jsonl')

# Default run size: 0 means run all pairs
MAX_FILES = 0

# Request pacing only (no retry logic)
API_SLEEP_BETWEEN_CALLS = 0.3

# Model choice (override via env var GEMINI_MODEL if desired)
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-2.5-pro')

assert LEAN_ROOT.exists(), f'Lean root not found: {LEAN_ROOT}'
assert HOL4_ROOT.exists(), f'HOL4 root not found: {HOL4_ROOT}'

def get_api_key() -> str:
    key = os.getenv('GEMINI_API_KEY')
    if not key:
        raise RuntimeError(
            'GEMINI_API_KEY is not set. In PowerShell run: $env:GEMINI_API_KEY = "..."'
        )
    return key

# --- Import / install google-generativeai ---
try:
    import google.generativeai as genai
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'google-generativeai'])
    import google.generativeai as genai

genai.configure(api_key=get_api_key())
model = genai.GenerativeModel(GEMINI_MODEL)
print('Configured Gemini model:', GEMINI_MODEL)

def extract_text_from_response(resp) -> str:
    if resp is None:
        return ''

    try:
        text = getattr(resp, 'text', None)
        if isinstance(text, str) and text.strip():
            return text.strip()
    except Exception:
        pass

    pieces: list[str] = []
    for cand in getattr(resp, 'candidates', []) or []:
        content = getattr(cand, 'content', None)
        for part in getattr(content, 'parts', []) or []:
            part_text = getattr(part, 'text', None)
            if isinstance(part_text, str) and part_text.strip():
                pieces.append(part_text.strip())

    return '\n'.join(pieces).strip()

def get_finish_reason_summary(resp) -> str:
    reasons: list[str] = []
    for idx, cand in enumerate(getattr(resp, 'candidates', []) or [], start=1):
        finish_reason = getattr(cand, 'finish_reason', None)
        finish_message = getattr(cand, 'finish_message', None)
        reasons.append(
            f'candidate {idx}: finish_reason={finish_reason}, finish_message={finish_message}'
        )
    return '; '.join(reasons) if reasons else 'no candidates returned'

Configured Gemini model: gemini-2.5-pro


In [2]:
@dataclass(frozen=True)
class FilePair:
    # Canonical pair key (used for caching/results)
    base: str
    hol4_path: Path
    lean_path: Path

def read_text_safely(path: Path) -> str:
    # Prefer UTF-8; replace undecodable bytes to keep pipeline robust
    return path.read_text(encoding='utf-8', errors='replace')

def truncate_middle(text: str, *, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    keep_head = max_chars // 2
    keep_tail = max_chars - keep_head
    return (
        text[:keep_head]
        + f"\n\n--- TRUNCATED ({len(text) - max_chars} chars omitted) ---\n\n"
        + text[-keep_tail:]
    )

def prepare_text_for_llm(path: Path) -> str:
    raw = read_text_safely(path)
    if USE_TRUNCATION:
        return truncate_middle(raw, max_chars=MAX_CHARS_PER_FILE)
    return raw

def find_hol4_scripts(root: Path) -> dict[str, Path]:
    scripts: dict[str, Path] = {}
    for p in root.rglob('*Script.sml'):
        base = p.name.removesuffix('Script.sml')
        key = base.lower()
        # If duplicates exist, keep the shortest path (more 'canonical')
        if key not in scripts or len(str(p)) < len(str(scripts[key])):
            scripts[key] = p
    return scripts

def find_lean_files(root: Path) -> dict[str, Path]:
    leans: dict[str, Path] = {}
    for p in root.rglob('*.lean'):
        key = p.stem.lower()
        if key not in leans or len(str(p)) < len(str(leans[key])):
            leans[key] = p
    return leans

# Special-case mapping: HOL4 base -> Lean base
SPECIAL_MATCHES: dict[str, str] = {
    'lprefix_lub': 'lprefixlub',
    'semantics_min': 'semanticsmin',
}

def collect_pairs(hol4_root: Path, lean_root: Path) -> list[FilePair]:
    hol4 = find_hol4_scripts(hol4_root)
    lean = find_lean_files(lean_root)

    pairs: list[FilePair] = []
    matched_hol4: set[str] = set()
    matched_lean: set[str] = set()

    # 1) Direct name matches
    for b in sorted(set(hol4).intersection(lean)):
        pairs.append(FilePair(base=b, hol4_path=hol4[b], lean_path=lean[b]))
        matched_hol4.add(b)
        matched_lean.add(b)

    # 2) Explicit special-case matches (HOL4 -> Lean)
    for hol4_base, lean_base in SPECIAL_MATCHES.items():
        if hol4_base in hol4 and lean_base in lean:
            if hol4_base in matched_hol4 or lean_base in matched_lean:
                continue
            pairs.append(
                FilePair(base=hol4_base, hol4_path=hol4[hol4_base], lean_path=lean[lean_base])
            )
            matched_hol4.add(hol4_base)
            matched_lean.add(lean_base)

    return pairs

hol4_map = find_hol4_scripts(HOL4_ROOT)
lean_map = find_lean_files(LEAN_ROOT)
pairs = collect_pairs(HOL4_ROOT, LEAN_ROOT)

matched_hol4_paths = {fp.hol4_path for fp in pairs}
matched_lean_paths = {fp.lean_path for fp in pairs}

hol4_unmatched = sorted(
    key for key, path in hol4_map.items()
    if path not in matched_hol4_paths
)
lean_unmatched = sorted(
    key for key, path in lean_map.items()
    if path not in matched_lean_paths
)

print('HOL4 scripts found:', len(hol4_map))
print('Lean files found:', len(lean_map))
print('Pairs found:', len(pairs))
print('Example pairs:')
for fp in pairs[:10]:
    print('-', fp.base)
    print('  HOL4:', fp.hol4_path)
    print('  Lean:', fp.lean_path)

print('\nUnmatched HOL4 scripts:', len(hol4_unmatched))
for key in hol4_unmatched[:20]:
    print('-', key)
    print('  HOL4:', hol4_map[key])

print('\nUnmatched Lean files:', len(lean_unmatched))
for key in lean_unmatched[:20]:
    print('-', key)
    print('  Lean:', lean_map[key])

if len(pairs) == 0:
    # Diagnostics: show a few keys to spot naming mismatches
    print('\nNo pairs found. Sample HOL4 bases:', list(sorted(hol4_map.keys()))[:20])
    print('Sample Lean bases:', list(sorted(lean_map.keys()))[:20])

HOL4 scripts found: 20
Lean files found: 21
Pairs found: 20
Example pairs:
- ast
  HOL4: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\hol4-ultramin\astScript.sml
  Lean: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\LeanTypeSound\Ast.lean
- evaluate
  HOL4: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\hol4-ultramin\evaluateScript.sml
  Lean: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\LeanTypeSound\Evaluate.lean
- evaluateprops
  HOL4: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\hol4-ultramin\evaluatePropsScript.sml
  Lean: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\LeanTypeSound\EvaluateProps.lean
- ffi
  HOL4: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\hol4-ultramin\ffiScript.sml
  Lean: D:\Computing\Dissertation\CML claude test\cakeml\lean_type_sound\LeanTypeSound\Ffi.lean
- fpsem
  HOL4: D:\Computing\Dissertation\CML claude test\cakeml\lea

In [3]:
def load_existing_results(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    existing: dict[str, dict] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            key = obj.get('base')
            if key:
                existing[key] = obj
    return existing

existing = load_existing_results(RESULTS_JSONL)
print('Existing cached results:', len(existing))

# Choose which pairs to run this session
todo = [p for p in pairs if p.base not in existing]
print('Pairs remaining:', len(todo))

# MAX_FILES <= 0 means run all remaining pairs
to_run = todo if MAX_FILES <= 0 else todo[:MAX_FILES]
print('Pairs selected for this run:', len(to_run))
print([p.base for p in to_run[:20]])
if len(to_run) > 20:
    print('... (truncated list display)')

Existing cached results: 19
Pairs remaining: 1
Pairs selected for this run: 1
['weakening']


In [4]:
# # Smoke test mode: run analysis on 1 pairs only
# # this is a debug cell, actual run just comment out whole cell
# SMOKE_TEST_COUNT = 1
# # Keep a copy of current selection so you can restore if needed
# to_run_full = list(to_run)

# to_run = todo[:SMOKE_TEST_COUNT]
# print(f'Smoke test enabled: {len(to_run)} pairs selected')
# print([p.base for p in to_run])

# # To disable smoke test later and restore full selection:
# # to_run = to_run_full

In [4]:
# prompts builder block

PRINT_PROMPT_TO_MODEL = True

def print_model_prompt(stage: str, base: str, prompt: str) -> None:
    if not PRINT_PROMPT_TO_MODEL:
        return
    print(f'[{stage}] prompt begin: {base}', flush=True)
    print(prompt, flush=True)
    print(f'[{stage}] prompt end: {base}', flush=True)

SYSTEM_INSTRUCTIONS = """\
You are reviewing a translation from HOL4/CakeML SML theory scripts to Lean 4 code.
Find genuine translation errors only. Missing/simplified Lean proofs are acceptable unless they change meaning.
If correct, output exactly RIGHT. Otherwise output STRICT JSON only.
"""

ERROR_SCHEMA_TEXT = """{
  \"base\": string,
  \"status\": \"error\",
  \"summary\": string,
  \"errors\": [{
    \"kind\": string,
    \"severity\": \"low\" | \"medium\" | \"high\",
    \"hol4_evidence\": string,
    \"lean_evidence\": string,
    \"why_wrong\": string
  }],
  \"confidence\": float (0.0 to 1.0),
  \"notes\": string
}"""

def read_pair_texts(hol4_path: Path, lean_path: Path) -> tuple[str, str]:
    hol4_text = hol4_path.read_text(encoding='utf-8', errors='replace')
    lean_text = lean_path.read_text(encoding='utf-8', errors='replace')
    return hol4_text, lean_text

def build_prompt(*, base: str, hol4_path: Path, lean_path: Path, hol4_text: str, lean_text: str) -> str:
    return f"""\
File pair base name: {base}


Task: compare HOL4 and Lean. Report only translation errors. You may report more than one error per pair if needed, but be concise.
Output contract:
- If no non-proof translation error: RIGHT
- Else: STRICT JSON with schema:
{ERROR_SCHEMA_TEXT}

--- HOL4 source text ---
{hol4_text}

--- Lean source text ---
{lean_text}
"""

def build_fix_prompt(*, base: str, hol4_path: Path, lean_path: Path, hol4_text: str, lean_text: str, diagnostic: dict) -> str:
    return f"""\
Produce corrected Lean code for this pair.
File pair base name: {base}
HOL4 source: {hol4_path}
Current Lean target: {lean_path}

Stage 1 diagnostic JSON:
{json.dumps(diagnostic, ensure_ascii=False)}

Requirements: fix translation errors only; no explanations; output Lean code only.

--- HOL4 source text ---
{hol4_text}

--- Current Lean source text ---
{lean_text}
"""

In [ ]:
def normalize_diagnostic_result(raw, *, base: str, hol4_path: Path, lean_path: Path) -> dict:
    if isinstance(raw, str):
        txt = raw.strip()
        if txt.upper() == 'RIGHT':
            return {
                'base': base,
                'status': 'right',
                'summary': 'Translation looks correct (proof omissions ignored).',
                'errors': [],
                'confidence': 0.8,
                'notes': '',
            }
        raise ValueError(f'Unexpected string response: {txt[:120]}')

    if not isinstance(raw, dict):
        raise TypeError(f'Unexpected result type: {type(raw)!r}')

    result = dict(raw)
    result.setdefault('base', base)
    result.setdefault('status', 'error')
    result.setdefault('summary', '')
    result.setdefault('errors', [])
    result.setdefault('confidence', 0.0)
    result.setdefault('notes', '')
    return result

def call_gemini_text(prompt: str, *, temperature: float = 0.1) -> str:
    resp = model.generate_content(
        prompt,
        generation_config={
            'temperature': temperature,
            'top_p': 0.95,
        },
    )
    text = extract_text_from_response(resp)
    if not text:
        raise RuntimeError(f'No text returned by model; {get_finish_reason_summary(resp)}')
    return text.strip()

def extract_lean_code(text: str) -> str:
    stripped = text.strip()
    if stripped.startswith('```'):
        lines = stripped.splitlines()
        if len(lines) >= 3 and lines[0].startswith('```') and lines[-1].strip() == '```':
            return '\n'.join(lines[1:-1]).strip()
    return stripped

def call_gemini_json_or_right(prompt: str):
    resp = model.generate_content(
        [SYSTEM_INSTRUCTIONS, prompt],
        generation_config={
            'temperature': 0.1,
            'top_p': 0.95,
        },
    )
    text = extract_text_from_response(resp)
    if not text:
        raise RuntimeError(f'No text returned by model; {get_finish_reason_summary(resp)}')

    stripped = text.strip()
    if stripped.upper() == 'RIGHT':
        return 'RIGHT'

    try:
        return json.loads(stripped)
    except json.JSONDecodeError as parse_err:
        start = stripped.find('{')
        end = stripped.rfind('}')
        if start != -1 and end != -1 and end > start:
            return json.loads(stripped[start : end + 1])
        raise RuntimeError(f'JSON parse failed: {parse_err}')

In [ ]:
def analyze_pair(fp: FilePair) -> dict:
    hol4_text, lean_text = read_pair_texts(fp.hol4_path, fp.lean_path)
    prompt = build_prompt(
        base=fp.base,
        hol4_path=fp.hol4_path,
        lean_path=fp.lean_path,
        hol4_text=hol4_text,
        lean_text=lean_text,
    )
    print_model_prompt('stage 1', fp.base, prompt)
    raw = call_gemini_json_or_right(prompt)
    result = normalize_diagnostic_result(
        raw,
        base=fp.base,
        hol4_path=fp.hol4_path,
        lean_path=fp.lean_path,
    )
    result['stage'] = 1
    return result

def append_jsonl(path: Path, obj: dict) -> None:
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False))
        f.write('\n')

# Run validation
run_t0 = time.time()
print(f'[stage 1] starting {len(to_run)} pair(s)')
for i, fp in enumerate(to_run, start=1):
    pair_t0 = time.time()
    print(f'[stage 1 {i}/{len(to_run)}] {fp.base}: read files', flush=True)
    try:
        print(f'[stage 1 {i}/{len(to_run)}] {fp.base}: call model', flush=True)
        out = analyze_pair(fp)
        print(f'[stage 1 {i}/{len(to_run)}] {fp.base}: write jsonl', flush=True)
        append_jsonl(RESULTS_JSONL, out)
        elapsed = time.time() - pair_t0
        print(f'[stage 1 {i}/{len(to_run)}] {fp.base}: done in {elapsed:.1f}s', flush=True)
        time.sleep(API_SLEEP_BETWEEN_CALLS)
    except Exception as e:
        err_obj = {
            'base': fp.base,
            'status': 'error',
            'summary': 'Error during stage 1 analysis call',
            'errors': [],
            'confidence': 0.0,
            'notes': str(e),
            'error_type': type(e).__name__,
            'error_traceback': traceback.format_exc(),
            'stage': 1,
        }
        append_jsonl(RESULTS_JSONL, err_obj)
        elapsed = time.time() - pair_t0
        print(f'[stage 1 {i}/{len(to_run)}] {fp.base}: ERROR after {elapsed:.1f}s -> {e}', flush=True)

print(f'[stage 1] done in {time.time() - run_t0:.1f}s. Output: {RESULTS_JSONL.resolve()}')

[stage 1] starting 1 pair(s)
[stage 1 1/1] weakening: read files
[stage 1 1/1] weakening: call model
[stage 1] prompt begin: weakening
File pair base name: weakening


Task: compare HOL4 and Lean. Report only translation errors. You may report more than one error per pair if needed, but be concise.
Output contract:
- If no non-proof translation error: RIGHT
- Else: STRICT JSON with schema:
{
  "base": string,
  "status": "error",
  "summary": string,
  "errors": [{
    "kind": string,
    "severity": "low" | "medium" | "high",
    "hol4_evidence": string,
    "lean_evidence": string,
    "why_wrong": string
  }],
  "confidence": number,
  "notes": string
}

--- HOL4 source text ---
(*
  Weakening lemmas used in type soundness
*)
Theory weakening
Ancestors
  option rich_list alist ast typeSystem typeSysProps
  namespaceProps semanticPrimitives typeSoundInvariants
Libs
  preamble

Definition weak_tenvE_def:
weak_tenvE tenv tenv' =
  (num_tvs tenv ≥ num_tvs tenv' ∧
   ∀n inc tvs t.
    (t

In [ ]:
# Generate fixed Lean files directly from stage 1 errors
FIXED_RESULTS_JSONL = Path('compare_validate_fixed_results.jsonl')
REFINED_LEAN_ROOT = Path('refined_lean_TS')
MAX_FIXES = MAX_FILES

pair_by_base = {p.base: p for p in pairs}
stage1_rows = load_existing_results(RESULTS_JSONL)

fix_source = [
    row for _, row in sorted(stage1_rows.items())
    if row.get('status') == 'error' and row.get('base') in pair_by_base and row.get('stage') == 1
]
to_fix = fix_source if MAX_FIXES <= 0 else fix_source[:MAX_FIXES]

print('Stage 1 rows available:', len(stage1_rows))
print('Stage 1 error rows selected for fix generation:', len(to_fix))

fix_t0 = time.time()
print(f'[fix] starting {len(to_fix)} pair(s)')
for i, row in enumerate(to_fix, start=1):
    pair_t0 = time.time()
    base = row['base']
    fp = pair_by_base[base]
    print(f'[fix {i}/{len(to_fix)}] {base}: read files', flush=True)

    try:
        hol4_text, lean_text = read_pair_texts(fp.hol4_path, fp.lean_path)
        print(f'[fix {i}/{len(to_fix)}] {base}: build prompt', flush=True)
        prompt = build_fix_prompt(
            base=base,
            hol4_path=fp.hol4_path,
            lean_path=fp.lean_path,
            hol4_text=hol4_text,
            lean_text=lean_text,
            diagnostic=row,
        )
        print_model_prompt('fix', base, prompt)
        print(f'[fix {i}/{len(to_fix)}] {base}: call model', flush=True)
        fixed_raw = call_gemini_text(prompt, temperature=0.1)
        fixed_lean = extract_lean_code(fixed_raw)

        out_path = REFINED_LEAN_ROOT / fp.lean_path.name
        out_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'[fix {i}/{len(to_fix)}] {base}: write file', flush=True)
        out_path.write_text(fixed_lean + '\n', encoding='utf-8')

        fix_obj = {
            'base': base,
            'status': 'fixed',
            'summary': 'Generated corrected Lean file from stage 1 diagnostic',
            'output_path': str(out_path),
            'diagnostic_stage': row.get('stage', 1),
        }
        append_jsonl(FIXED_RESULTS_JSONL, fix_obj)
        elapsed = time.time() - pair_t0
        print(f'[fix {i}/{len(to_fix)}] {base}: done in {elapsed:.1f}s -> {out_path}', flush=True)
        time.sleep(API_SLEEP_BETWEEN_CALLS)
    except Exception as e:
        err_obj = {
            'base': base,
            'status': 'error',
            'summary': 'Error during fix generation',
            'output_path': '',
            'notes': str(e),
            'error_type': type(e).__name__,
            'error_traceback': traceback.format_exc(),
        }
        append_jsonl(FIXED_RESULTS_JSONL, err_obj)
        elapsed = time.time() - pair_t0
        print(f'[fix {i}/{len(to_fix)}] {base}: ERROR after {elapsed:.1f}s -> {e}', flush=True)

print(f'[fix] done in {time.time() - fix_t0:.1f}s. Output: {FIXED_RESULTS_JSONL.resolve()}')
print('Fixed Lean files root:', REFINED_LEAN_ROOT.resolve())

Stage 1 rows available: 20
Stage 1 error rows selected for fix generation: 7
[fix] starting 7 pair(s)
[fix 1/7] evaluate: read files
[fix 1/7] evaluate: build prompt
[fix] prompt begin: evaluate
Produce corrected Lean code for this pair.
File pair base name: evaluate
HOL4 source: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\evaluateScript.sml
Current Lean target: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound\Evaluate.lean

Stage 1 diagnostic JSON:
{"base": "evaluate", "status": "error", "summary": "The Lean translation of the `evaluate` function is incomplete. It omits the semantics for `Force` and `EvalOp` applications, which are part of the CakeML language definition.", "errors": [{"kind": "Missing implementation", "severity": "high", "hol4_evidence": "The HOL4 definition for `evaluate st env [App op es]` includes cases for `Force` and `EvalOp` operator classes, defining the semantics for thunks and dynamic evaluation.\n\n```sml\n  eva

In [1]:
import json
from pathlib import Path

src = Path("compare_validate_results.jsonl")
dst = Path("compare_validate_errors.jsonl")

if not src.exists():
    raise FileNotFoundError(f"Input file not found: {src.resolve()}")

kept = 0
total = 0

with src.open("r", encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
    for line in fin:
        line = line.strip()
        if not line:
            continue
        total += 1
        obj = json.loads(line)
        if str(obj.get("status", "")).lower() == "error":
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            kept += 1

print(f"Read {total} rows from {src}.")
print(f"Wrote {kept} error rows to {dst.resolve()}.")

Read 20 rows from compare_validate_results.jsonl.
Wrote 7 error rows to E:\NUS\mcomp\Dissertation\CakeML_data_extraction\compare_validate_errors.jsonl.


In [ ]:
# Summarize stage 1 and fixed outputs
try:
    import pandas as pd
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas'])
    import pandas as pd

FIXED_RESULTS_JSONL = Path('compare_validate_fixed_results.jsonl')

stage1_rows = list(load_existing_results(RESULTS_JSONL).values())
fixed_rows = list(load_existing_results(FIXED_RESULTS_JSONL).values())

print('Stage 1 rows:', len(stage1_rows))
print('Fixed rows:', len(fixed_rows))

if len(stage1_rows) == 0:
    print('No stage 1 results yet.')
else:
    df0 = pd.DataFrame(stage1_rows)
    cols0 = [c for c in ['base', 'stage', 'status', 'confidence', 'summary'] if c in df0.columns]
    display(df0[cols0].sort_values(['stage', 'status', 'confidence'], ascending=[True, True, True]).head(50))
    display(df0['status'].value_counts(dropna=False))
    if 'stage' in df0.columns:
        display(df0['stage'].value_counts(dropna=False))

if len(fixed_rows) == 0:
    print('No fixed outputs yet.')
else:
    df2 = pd.DataFrame(fixed_rows)
    cols2 = [c for c in ['base', 'status', 'summary', 'output_path'] if c in df2.columns]
    display(df2[cols2].sort_values(['status', 'base'], ascending=[True, True]).head(50))
    display(df2['status'].value_counts(dropna=False))

Stage 1 rows: 20
Fixed rows: 7


,base,stage,status,confidence,summary,hol4_path,lean_path
11,semanticprimitives,1,error,0.95,The function `v_to_char_list` is translated in...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
1,evaluate,1,error,5.00,The Lean translation of the `evaluate` functio...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
7,mllist,1,error,5.00,The Lean `sort` function has an incorrect sign...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
12,semanticprimitivesprops,1,error,5.00,The definition of free variables for `Letrec` ...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
14,typesoundinvariants,1,error,5.00,The translation of the inductive predicate `ty...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
15,typesysprops,1,error,5.00,A theorem about namespace lookups (`nsLookup_a...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
17,weakening,1,error,5.00,The theorem `type_s_weakening` has an incorrec...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
0,ast,1,right,0.80,Translation looks correct (proof omissions ign...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
2,evaluateprops,1,right,0.80,Translation looks correct (proof omissions ign...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
3,ffi,1,right,0.80,Translation looks correct (proof omissions ign...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...


status
right    13
error     7
Name: count, dtype: int64

stage
1    20
Name: count, dtype: int64

,base,status,summary,output_path,hol4_path,lean_path
0,evaluate,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\Evaluate.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
1,mllist,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\Mllist.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
2,semanticprimitives,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\SemanticPrimitives.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
3,semanticprimitivesprops,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\SemanticPrimitivesProps.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
4,typesoundinvariants,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\TypeSoundInvariants.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
5,typesysprops,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\TypeSysProps.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...
6,weakening,fixed,Generated corrected Lean file from stage 1 dia...,refined_lean_TS\Weakening.lean,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...,E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_t...


status
fixed    7
Name: count, dtype: int64